# Model 1 — LightGBM on engineered features

Predicts the **6-vector of compression perplexities** from prompt text.

- Data pulled live from the GitHub repo (`perplexity_wide_complete.csv`).
- Targets modeled in **log space**; predictions inverted to raw perplexity for reporting.
- **Stratified random 70/30 split by dataset** (each dataset: 70% train / 30% test), `random_state=42`.
- Trained model saved to `artifacts/`.

Run top to bottom. Requires `kv_common.py` in the same folder.

## Colab setup

Run this cell **first**. It clones the repo (so `kv_common.py` is available),
installs packages, and optionally mounts Google Drive so your cached embeddings
and trained models survive a disconnect.

> For notebook 04 (LoRA), also enable a GPU: **Runtime → Change runtime type → T4 GPU**.

In [1]:
# Clone the repo so kv_common.py and artifacts live in one place.
import os
if not os.path.exists("KVCacheCompression"):
    !git clone -q https://github.com/yoshikodes/KVCacheCompression.git
# Work inside the notebooks folder (edit if your notebooks live elsewhere).
if os.path.basename(os.getcwd()) != "notebooks":
    %cd KVCacheCompression/notebooks
print("cwd:", os.getcwd())
assert os.path.exists("kv_common.py"), "kv_common.py not found"


cwd: /mnt/user-data/outputs/notebooks


In [2]:
# Install packages not preinstalled on Colab.
# If you get an import error right after this, do Runtime -> Restart, then re-run from the top.
!pip install -q lightgbm scikit-learn scipy requests joblib

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [3]:
import numpy as np, pandas as pd, os, joblib
import kv_common as kv

os.makedirs("artifacts", exist_ok=True)

df = kv.load_data()
train_df, test_df = kv.stratified_split(df)

LOG_SPACE = True
y_train = kv.get_targets(train_df, log_space=LOG_SPACE)
y_test  = kv.get_targets(test_df,  log_space=LOG_SPACE)

# Mean baseline for reference in every notebook.
baseline = kv.baseline_predict_mean(y_train, len(y_test))
baseline_metrics = kv.evaluate(y_test, baseline)
print("Mean-baseline OVERALL MAE_log:",
      round(baseline_metrics[baseline_metrics.setting=='OVERALL'].MAE_log.iloc[0], 4))


Loaded data from: https://raw.githubusercontent.com/yoshikodes/KVCacheCompression/main/perplexity_data/perplexity_wide_complete.csv


Rows: 3153 -> 3150 after dropping NaN targets and dup prompts.
Per-dataset counts:
dataset
hellaswag        1023
arc_challenge    1022
gsm8k             963
wikitext103       142
Train: 2204  Test: 946
Train per-dataset: {'hellaswag': 716, 'arc_challenge': 715, 'gsm8k': 674, 'wikitext103': 99}
Test  per-dataset: {'arc_challenge': 307, 'hellaswag': 307, 'gsm8k': 289, 'wikitext103': 43}
Mean-baseline OVERALL MAE_log: 1.1154


/mnt/user-data/outputs/notebooks/kv_common.py:193: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "Spearman": spearmanr(yt, yp).correlation,
/mnt/user-data/outputs/notebooks/kv_common.py:193: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "Spearman": spearmanr(yt, yp).correlation,
/mnt/user-data/outputs/notebooks/kv_common.py:193: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "Spearman": spearmanr(yt, yp).correlation,
/mnt/user-data/outputs/notebooks/kv_common.py:193: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "Spearman": spearmanr(yt, yp).correlation,
/mnt/user-data/outputs/notebooks/kv_common.py:193: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  "Spearman": spearmanr(yt, yp).correlation,
/mnt/user-data/outputs/notebooks/kv_common.py:193:

## Feature engineering
Cheap surface features of the prompt text.

In [4]:
import re

def extract_features(prompt: str) -> dict:
    s = str(prompt)
    toks = s.split()
    n_tok = max(len(toks), 1)
    n_char = max(len(s), 1)
    digits = sum(c.isdigit() for c in s)
    puncts = sum(not c.isalnum() and not c.isspace() for c in s)
    uppers = sum(c.isupper() for c in s)
    numbers = re.findall(r"\d+\.?\d*", s)
    return {
        "n_char": len(s),
        "n_tok": len(toks),
        "n_sent": s.count(".") + s.count("?") + s.count("!") + 1,
        "avg_tok_len": n_char / n_tok,
        "type_token_ratio": len(set(toks)) / n_tok,
        "digit_ratio": digits / n_char,
        "punct_ratio": puncts / n_char,
        "upper_ratio": uppers / n_char,
        "num_count": len(numbers),
        "has_question": int("?" in s),
        "mean_word_len": np.mean([len(t) for t in toks]) if toks else 0.0,
        "max_word_len": max((len(t) for t in toks), default=0),
    }

def feature_matrix(prompts):
    return pd.DataFrame([extract_features(p) for p in prompts])

X_train = feature_matrix(train_df["prompt"])
X_test  = feature_matrix(test_df["prompt"])
FEATURE_NAMES = X_train.columns.tolist()
print("features:", FEATURE_NAMES)
X_train.head()

features: ['n_char', 'n_tok', 'n_sent', 'avg_tok_len', 'type_token_ratio', 'digit_ratio', 'punct_ratio', 'upper_ratio', 'num_count', 'has_question', 'mean_word_len', 'max_word_len']


,n_char,n_tok,n_sent,avg_tok_len,type_token_ratio,digit_ratio,punct_ratio,upper_ratio,num_count,has_question,mean_word_len,max_word_len
0,147,29,5,5.068966,0.896552,0.034014,0.027211,0.027211,4,1,4.000000,8
1,65,10,1,6.500000,1.000000,0.000000,0.000000,0.015385,0,0,5.600000,13
2,260,43,5,6.046512,0.860465,0.000000,0.038462,0.023077,0,0,5.069767,16
3,240,41,6,5.853659,0.731707,0.020833,0.033333,0.029167,2,1,4.878049,11
4,322,59,5,5.457627,0.813559,0.000000,0.031056,0.021739,0,0,4.474576,9


## Train
One LightGBM regressor per output via `MultiOutputRegressor`.

In [5]:
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor

base = lgb.LGBMRegressor(
    n_estimators=400, num_leaves=31, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
    random_state=kv.RANDOM_STATE, verbose=-1,
)
model = MultiOutputRegressor(base)
model.fit(X_train.values, y_train)
print("trained.")

trained.


## Evaluate

In [6]:
pred = model.predict(X_test.values)
metrics = kv.evaluate(y_test, pred)
print(metrics.round(4).to_string(index=False))
kv.print_comparison(metrics, baseline_metrics)

     setting  MAE_log  RMSE_log  R2_log  Spearman   MAE_raw
      h2o_20   0.8170    1.4435  0.4834    0.8197 1465.7773
      h2o_40   0.6551    1.1228  0.4378    0.7402  180.7043
      h2o_60   0.6408    1.0870  0.4256    0.7198  111.7889
kvquant_2bit   0.6009    1.0493  0.4331    0.7276   81.8392
kvquant_3bit   0.6665    1.1656  0.4025    0.7209  565.4954
kvquant_4bit   0.6474    1.0932  0.4250    0.7195  100.3351
     OVERALL   0.6713    1.1602  0.4346    0.7413  417.6567

=== Model vs. mean-baseline (OVERALL, log space) ===
  MAE_log   model=0.6713   baseline=1.1154   BEATS baseline
  R2_log    model=0.4346   baseline=-0.0003


## Feature importance (mean gain across the 6 outputs)

In [7]:
imp = np.zeros(len(FEATURE_NAMES))
for est in model.estimators_:
    imp += est.feature_importances_
imp_df = pd.DataFrame({"feature": FEATURE_NAMES, "importance": imp}
                     ).sort_values("importance", ascending=False)
print(imp_df.to_string(index=False))

         feature  importance
   mean_word_len     11251.0
     upper_ratio     10743.0
     punct_ratio     10431.0
     avg_tok_len      9394.0
          n_char      8139.0
type_token_ratio      7305.0
           n_tok      4926.0
    max_word_len      3913.0
          n_sent      1842.0
     digit_ratio      1795.0
    has_question      1501.0
       num_count       760.0


## Save model

In [8]:
joblib.dump({"model": model, "feature_names": FEATURE_NAMES,
             "settings": kv.SETTINGS, "log_space": LOG_SPACE},
            "artifacts/model1_lightgbm.joblib")
metrics.to_csv("artifacts/model1_lightgbm_metrics.csv", index=False)
print("saved artifacts/model1_lightgbm.joblib")

saved artifacts/model1_lightgbm.joblib
